# Per-Residue ESM-2 Conditioned Flow Matching

$$p(\phi, \psi \mid h_i^{\text{ESM2}})$$

ESM-2 per-residue hidden states already encode amino acid identity, local sequence context,
and evolutionary information. No separate amino acid embedding is needed.

**Pipeline:**
1. Download all ~1019 PDB files
2. For each residue: extract `(phi, psi)` and its ESM-2 hidden state `h_i` at `seq_pos + 1`
3. Cache the dataset
4. Train `ConditionedFlow` from `models.py`

## 1. Imports

In [ ]:
import glob
import os
import pickle as pkl
import random

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm

from models import ConditionedFlow
from pdb_loading import get_residues_with_positions, batch_download_pdb

AA_LETTERS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX  = {aa: i for i, aa in enumerate(AA_LETTERS)}
device = 'cpu'
print('Device:', device)

## 2. Load ESM-2

In [ ]:
esm_model = pkl.load(open('models/esm2_8M.pkl', 'rb'))
tokenizer  = pkl.load(open('models/esm2_8M_tokenizer.pkl', 'rb'))
esm_model.eval()
print('ESM-2 8M loaded')

## 3. Download All PDB Files

Uses the protein IDs from `protein_features_with_analysis.pkl` (~1019 proteins).
Already-downloaded files are skipped automatically.

In [ ]:
protein_dict = pkl.load(open('protein_features_with_analysis.pkl', 'rb'))
all_pdb_ids  = list(protein_dict.keys())
print(f'Target proteins: {len(all_pdb_ids)}')

already = {f.split('/')[-1].replace('.pdb', '') for f in glob.glob('PDBs/*.pdb')}
to_download = [pid for pid in all_pdb_ids if pid.upper() not in {k.upper() for k in already}]
print(f'Already downloaded: {len(already)}   Still needed: {len(to_download)}')

if to_download:
    successful, failed = batch_download_pdb(to_download, download_dir='PDBs')
    print(f'New downloads: {len(successful)}  Failed: {len(failed)}')
else:
    print('All files already present.')

## 4. Build Dataset

For every residue in every PDB file:
- Extract `(phi, psi)` angles (interior residues only)
- Extract the ESM-2 hidden state at `hidden_states[seq_pos + 1]`

The dataset is cached to `dataset_cache.pkl` so this only runs once.

In [ ]:
CACHE_PATH = 'dataset_cache.pkl'

if os.path.exists(CACHE_PATH):
    print('Loading cached dataset...')
    cache = pkl.load(open(CACHE_PATH, 'rb'))
    angles_tensor = cache['angles']
    esm2_tensor   = cache['esm2']
    all_pdb_ids_r = cache['pdb_ids']
    all_aa_chars  = cache['aa_chars']
    print(f'Loaded {len(angles_tensor)} residues from cache.')
else:
    pdb_files = sorted(glob.glob('PDBs/*.pdb'))
    print(f'Building dataset from {len(pdb_files)} PDB files...')

    raw_angles, raw_esm2, raw_pdb_ids, raw_aa_chars = [], [], [], []

    with torch.no_grad():
        for pdb_path in tqdm(pdb_files):
            pdb_id = os.path.basename(pdb_path).replace('.pdb', '')
            try:
                full_seq, records = get_residues_with_positions(pdb_path)
            except Exception:
                continue
            if not records or not full_seq:
                continue

            inputs = tokenizer(full_seq, return_tensors='pt').to(device)
            hidden = esm_model(**inputs).last_hidden_state[0]  # (L+2, 320)

            for r in records:
                if r['aa'] not in AA_TO_IDX:
                    continue
                raw_angles.append([r['phi'], r['psi']])
                raw_esm2.append(hidden[r['seq_pos'] + 1])  # +1 for CLS
                raw_pdb_ids.append(pdb_id)
                raw_aa_chars.append(r['aa'])

    angles_tensor = torch.tensor(raw_angles, dtype=torch.float32) / 180.0
    esm2_tensor   = torch.stack(raw_esm2).float()
    all_pdb_ids_r = raw_pdb_ids
    all_aa_chars  = raw_aa_chars

    pkl.dump({
        'angles':   angles_tensor,
        'esm2':     esm2_tensor,
        'pdb_ids':  all_pdb_ids_r,
        'aa_chars': all_aa_chars,
    }, open(CACHE_PATH, 'wb'))
    print(f'Dataset built and cached: {len(angles_tensor)} residues.')

print(f'\nangles: {angles_tensor.shape}')
print(f'esm2:   {esm2_tensor.shape}')

In [ ]:
n_proteins = len(set(all_pdb_ids_r))
aa_counts  = Counter(all_aa_chars)
print(f'Proteins: {n_proteins}   Total residues: {len(angles_tensor)}')
print(f'Mean residues/protein: {len(angles_tensor)/n_proteins:.0f}')
print('\nPer-amino-acid counts:')
for aa in AA_LETTERS:
    bar = '#' * (aa_counts.get(aa, 0) // 200)
    print(f'  {aa}  {aa_counts.get(aa,0):6d}  {bar}')

## 5. Train / Test Split (held-out proteins)

In [ ]:
random.seed(42)
all_proteins  = sorted(set(all_pdb_ids_r))
test_proteins = set(random.sample(all_proteins, max(1, len(all_proteins) // 10)))

train_mask = torch.tensor([pid not in test_proteins for pid in all_pdb_ids_r])
test_mask  = ~train_mask

tr_ang  = angles_tensor[train_mask]
tr_esm2 = esm2_tensor[train_mask]

te_ang   = angles_tensor[test_mask]
te_esm2  = esm2_tensor[test_mask]
te_pids  = [pid for pid, m in zip(all_pdb_ids_r, test_mask.tolist()) if m]
te_aas   = [aa  for aa,  m in zip(all_aa_chars,  test_mask.tolist()) if m]

print(f'Train: {tr_ang.shape[0]:,} residues  ({len(all_proteins)-len(test_proteins)} proteins)')
print(f'Test:  {te_ang.shape[0]:,} residues  ({len(test_proteins)} proteins)')

## 6. Model

```
input: [ t (1) | x_t (2) | esm2 (320) ]  →  4 × 512 hidden  →  velocity (2)
```

ESM-2 per-residue embeddings encode amino acid identity, evolutionary context, and
predicted secondary structure — no separate amino acid embedding is needed.

In [ ]:
flow = ConditionedFlow(dim=2, embed_dim=320, h=512)
print(f'Parameters: {sum(p.numel() for p in flow.parameters()):,}')

## 7. Training

$$\mathcal{L} = \mathbb{E}_{t,\, x_0,\, x_1,\, h_i} \left[\| v_\theta(t,\, x_t,\, h_i) - (x_1 - x_0) \|^2\right]$$

In [ ]:
ITERATIONS  = 30000
BATCH_SIZE  = 1024
LR          = 1e-3
PRINT_EVERY = 2000

optimizer    = torch.optim.Adam(flow.parameters(), lr=LR)
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ITERATIONS)
loss_fn      = nn.MSELoss()
n_train      = tr_ang.shape[0]
loss_history = []

flow.train()
for i in range(ITERATIONS):
    idx  = torch.randint(0, n_train, (BATCH_SIZE,))
    x_1  = tr_ang[idx]    # (B, 2)
    emb  = tr_esm2[idx]   # (B, 320)

    x_0  = torch.randn_like(x_1)
    t    = torch.rand(BATCH_SIZE, 1)
    x_t  = (1 - t) * x_0 + t * x_1
    dx_t = x_1 - x_0

    optimizer.zero_grad()
    loss = loss_fn(flow(t, x_t, emb), dx_t)
    loss.backward()
    optimizer.step()
    scheduler.step()

    loss_history.append(loss.item())
    if (i + 1) % PRINT_EVERY == 0:
        lr_now = scheduler.get_last_lr()[0]
        print(f'iter {i+1:6d} | loss {loss.item():.4f} | lr {lr_now:.2e}')

print('Training complete.')

In [ ]:
plt.figure(figsize=(9, 3))
w = 200
plt.plot(np.convolve(loss_history, np.ones(w)/w, mode='valid'))
plt.xlabel('Iteration'); plt.ylabel('MSE Loss')
plt.title('Training Loss (smoothed)')
plt.tight_layout(); plt.show()

## 8. Generation Helpers

In [ ]:
@torch.no_grad()
def generate(flow_model, esm2: torch.Tensor,
             n_gen: int = 200, n_steps: int = 20) -> np.ndarray:
    """
    Generate n_gen (phi, psi) samples for each row in esm2.

    esm2   : (B, 320)  per-residue ESM-2 hidden states
    Returns: (B * n_gen, 2) in degrees
    """
    flow_model.eval()
    emb = esm2.repeat_interleave(n_gen, dim=0)   # (B*n_gen, 320)
    x   = torch.randn(emb.shape[0], 2)
    ts  = torch.linspace(0, 1.0, n_steps + 1)
    for i in range(n_steps):
        x = flow_model.step(x, ts[i], ts[i + 1], emb)
    return x.numpy() * 180.0

print('Generation helper ready.')

## 9. Demo A — Full Protein Ramachandran

Generate 50 samples per residue for a held-out protein and overlay on ground truth.

In [ ]:
test_pid = sorted(test_proteins)[0]
pid_mask = torch.tensor([pid == test_pid for pid in te_pids])

pid_esm2 = te_esm2[pid_mask]
gt_deg   = te_ang[pid_mask].numpy() * 180.0
print(f'Protein {test_pid}: {pid_esm2.shape[0]} interior residues')

gen_all = generate(flow, pid_esm2, n_gen=50)  # (L*50, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Ramachandran Plot — Protein {test_pid}', fontsize=13)

axes[0].scatter(gt_deg[:, 0], gt_deg[:, 1], s=20, color='steelblue')
axes[0].set_title(f'Ground Truth  ({len(gt_deg)} residues)')

axes[1].scatter(gen_all[:, 0], gen_all[:, 1], s=2, alpha=0.15, color='tomato')
axes[1].scatter(gt_deg[:, 0],  gt_deg[:, 1],  s=25, color='steelblue', zorder=3, label='GT')
axes[1].set_title('Generated (50 samples/residue) + GT overlay')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.axhline(0, color='k', lw=0.4, ls='--'); ax.axvline(0, color='k', lw=0.4, ls='--')
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
    ax.set_xlabel('φ (deg)'); ax.set_ylabel('ψ (deg)')

plt.tight_layout(); plt.show()

## 10. Demo B — Context Shifts the Distribution

Same amino acid, different proteins → different ESM-2 context → different predicted location.

In [ ]:
FOCUS_AA  = 'L'
N_SHOW    = 6
N_GEN_CTX = 400

seen_pids = set()
contexts  = []
for j, (pid, aa) in enumerate(zip(te_pids, te_aas)):
    if aa == FOCUS_AA and pid not in seen_pids:
        contexts.append({'pid': pid, 'esm2': te_esm2[j], 'gt': te_ang[j].numpy() * 180.0})
        seen_pids.add(pid)
contexts = contexts[:N_SHOW]

fig, axes = plt.subplots(2, 3, figsize=(13, 9), sharex=True, sharey=True)
fig.suptitle(f'{FOCUS_AA} (Leucine) — Same AA, Different Protein Contexts', fontsize=13)

for ax, ctx in zip(axes.flat, contexts):
    gen = generate(flow, ctx['esm2'].unsqueeze(0), n_gen=N_GEN_CTX)
    ax.scatter(gen[:, 0], gen[:, 1], s=6, alpha=0.4, color='tomato')
    ax.scatter([ctx['gt'][0]], [ctx['gt'][1]], s=100, color='steelblue', zorder=5, marker='*')
    ax.set_title(f'Protein {ctx["pid"]}', fontsize=10)
    ax.axhline(0, color='k', lw=0.4, ls='--'); ax.axvline(0, color='k', lw=0.4, ls='--')
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)

for ax in axes[-1, :]: ax.set_xlabel('φ (deg)')
for ax in axes[:, 0]:  ax.set_ylabel('ψ (deg)')
fig.legend(
    handles=[
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato',    markersize=8,  label='Generated'),
        plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='steelblue', markersize=12, label='Ground Truth'),
    ], loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## 11. Demo C — Per-Amino-Acid Distributions

Pool all test-set residues of each amino acid type and compare generated vs ground truth.
ESM-2 captures amino acid identity implicitly — no explicit type label used.

In [ ]:
N_GEN_AA = 5  # samples per test residue

gen_by_aa = {}
gt_by_aa  = {}
for aa in AA_LETTERS:
    mask = torch.tensor([c == aa for c in te_aas])
    if mask.sum() == 0:
        gen_by_aa[aa] = gt_by_aa[aa] = np.empty((0, 2))
        continue
    gt_by_aa[aa]  = te_ang[mask].numpy() * 180.0
    gen_by_aa[aa] = generate(flow, te_esm2[mask], n_gen=N_GEN_AA)

fig, axes = plt.subplots(4, 5, figsize=(18, 15), sharex=True, sharey=True)
fig.suptitle('Per-Amino-Acid Ramachandran — ESM-2 Conditioned Flow\n'
             'blue = ground truth, red = generated', fontsize=13)

for ax, aa in zip(axes.flat, AA_LETTERS):
    gt  = gt_by_aa[aa]
    gen = gen_by_aa[aa]
    if len(gt):  ax.scatter(gt[:,0],  gt[:,1],  s=12, alpha=0.8, color='steelblue')
    if len(gen): ax.scatter(gen[:,0], gen[:,1], s=4,  alpha=0.3, color='tomato')
    ax.set_title(f'{aa}  (n={len(gt)})', fontsize=10)
    ax.axhline(0, color='k', lw=0.4, ls='--'); ax.axvline(0, color='k', lw=0.4, ls='--')
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)

for ax in axes[-1, :]: ax.set_xlabel('φ (deg)')
for ax in axes[:, 0]:  ax.set_ylabel('ψ (deg)')
fig.legend(handles=[
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', markersize=8, label='Ground Truth'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato',    markersize=8, label='Generated'),
], loc='lower right', fontsize=11)
plt.tight_layout(); plt.show()

## 12. Demo D — ODE Trajectory for a Single Residue

Watch Gaussian noise transform into the predicted Ramachandran location for one specific residue.

In [ ]:
# Pick first Glycine in test set
traj_idx = next(j for j, aa in enumerate(te_aas) if aa == 'G')
res_esm2 = te_esm2[traj_idx]
res_gt   = te_ang[traj_idx].numpy() * 180.0
res_pid  = te_pids[traj_idx]
print(f'Residue G in {res_pid}  GT=({res_gt[0]:.1f}°, {res_gt[1]:.1f}°)')

N_TRAJ  = 2000
N_STEPS = 8
emb_rep = res_esm2.unsqueeze(0).expand(N_TRAJ, -1)
x       = torch.randn(N_TRAJ, 2)
ts      = torch.linspace(0, 1.0, N_STEPS + 1)

fig, axes = plt.subplots(1, N_STEPS + 1, figsize=(3.5*(N_STEPS+1), 3.5), sharex=True, sharey=True)
fig.suptitle(f'ODE Trajectory — Glycine in {res_pid}', fontsize=11)

def plot_step(ax, pts, t_label):
    ax.scatter(pts[:, 0]*180, pts[:, 1]*180, s=3, alpha=0.25, color='steelblue')
    ax.scatter([res_gt[0]], [res_gt[1]], s=80, color='red', zorder=5, marker='*')
    ax.set_title(f't={t_label:.2f}', fontsize=9)
    ax.axhline(0, color='k', lw=0.4, ls='--'); ax.axvline(0, color='k', lw=0.4, ls='--')
    ax.set_xlim(-270, 270); ax.set_ylim(-270, 270)

flow.eval()
with torch.no_grad():
    plot_step(axes[0], x.numpy(), 0.0)
    for i in range(N_STEPS):
        x = flow.step(x, ts[i], ts[i+1], emb_rep)
        plot_step(axes[i+1], x.numpy(), ts[i+1].item())

for ax in axes: ax.set_xlabel('φ'); ax.set_aspect('equal')
axes[0].set_ylabel('ψ')
plt.tight_layout(); plt.show()

## 13. NeRF Structure Reconstruction → PDB Export

Given generated torsion angles `(φ, ψ)` for each residue, the **Natural Extension Reference Frame (NeRF)** algorithm reconstructs backbone 3D coordinates one atom at a time using standard bond lengths and angles.

**Atom placement order per residue:**
```
N_i → CA_i → C_i → O_i
```
**Torsion angles used:**
| Placed atom | Dihedral used | Value |
|-------------|--------------|-------|
| N_i | N_{i-1}–CA_{i-1}–C_{i-1}–**N_i** | ψ_{i-1} |
| CA_i | CA_{i-1}–C_{i-1}–N_i–**CA_i** | ω ≈ 180° (trans peptide) |
| C_i | C_{i-1}–N_i–CA_i–**C_i** | φ_i |
| O_i | N_i–CA_i–C_i–**O_i** | ψ_i + 180° |

**Pipeline:** sequence → ESM-2 embeddings → flow matching (one φ/ψ per residue) → NeRF → PDB

In [ ]:
import math

# ── Standard backbone geometry (AMBER/CHARMM) ───────────────────────────────
BL = {          # Bond lengths (Å)
    'n_ca': 1.458,   # N–CA
    'ca_c': 1.525,   # CA–C
    'c_n' : 1.329,   # C–N  (peptide bond)
    'c_o' : 1.229,   # C=O  (carbonyl)
}
BA = {          # Bond angles (degrees)
    'n_ca_c': 111.2,  # N–CA–C
    'ca_c_n': 116.2,  # CA–C–N  (places next N)
    'c_n_ca': 121.7,  # C–N–CA  (places next CA)
    'ca_c_o': 120.8,  # CA–C=O  (places O)
}


def nerf(a, b, c, bond_length, bond_angle_deg, dihedral_deg):
    """
    Natural Extension Reference Frame: place atom D given A, B, C.

    The dihedral A–B–C–D = dihedral_deg and the bond angle B–C–D =
    bond_angle_deg define D's position at distance bond_length from C.

    Parameters
    ----------
    a, b, c        : np.ndarray (3,), coordinates of three anchor atoms
    bond_length    : float, |C–D| in Å
    bond_angle_deg : float, angle B–C–D in degrees
    dihedral_deg   : float, dihedral A–B–C–D in degrees

    Returns
    -------
    d : np.ndarray (3,), coordinates of the new atom D
    """
    theta = math.radians(bond_angle_deg)
    xi    = math.radians(dihedral_deg)

    bc     = c - b
    bc_hat = bc / np.linalg.norm(bc)

    n = np.cross(b - a, bc_hat)
    n_norm = np.linalg.norm(n)
    if n_norm < 1e-8:
        # Degenerate: A, B, C nearly collinear — pick an arbitrary perpendicular
        perp = np.array([0., 0., 1.]) if abs(bc_hat[2]) < 0.9 else np.array([1., 0., 0.])
        n = np.cross(bc_hat, perp)
        n /= np.linalg.norm(n)
    else:
        n /= n_norm

    m = np.cross(bc_hat, n)   # completes the right-handed frame

    d = (bond_length * (-math.cos(theta)) * bc_hat
         + bond_length *  math.sin(theta) * math.cos(xi) * m
         + bond_length *  math.sin(theta) * math.sin(xi) * n)

    return c + d


def build_backbone(sequence, phi_psi_deg, omega_deg=180.0):
    """
    Reconstruct backbone 3D coordinates from φ/ψ torsion angles via NeRF.

    Terminal residues (first φ, last ψ) are set by the caller — the model
    generates them freely and NeRF handles them without special casing.

    Parameters
    ----------
    sequence    : str, amino acid sequence of length L
    phi_psi_deg : np.ndarray (L, 2), columns = [φ, ψ] in degrees
    omega_deg   : float, peptide bond torsion (default 180° for trans)

    Returns
    -------
    coords : np.ndarray (L, 4, 3)
        Backbone atom coordinates in Å, axis-1 order: [N, CA, C, O]
    """
    L = len(sequence)
    assert phi_psi_deg.shape == (L, 2), \
        f"phi_psi_deg must be ({L}, 2), got {phi_psi_deg.shape}"

    coords = np.zeros((L, 4, 3), dtype=np.float64)

    # ── Seed residue 0 in a local Cartesian frame ────────────────────────────
    # N0 at origin, CA0 along +x, C0 in the xy-plane.
    ang_nca_c = math.radians(BA['n_ca_c'])
    coords[0, 0] = [0.0, 0.0, 0.0]                           # N0
    coords[0, 1] = [BL['n_ca'], 0.0, 0.0]                    # CA0
    # C0: angle N0–CA0–C0 = 111.2°
    coords[0, 2] = coords[0, 1] + BL['ca_c'] * np.array([
        math.cos(math.pi - ang_nca_c),
        math.sin(math.pi - ang_nca_c),
        0.0,
    ])                                                         # C0
    # O0: dihedral N0–CA0–C0–O0 ≈ ψ0 + 180° (O is anti to next N)
    coords[0, 3] = nerf(
        coords[0, 0], coords[0, 1], coords[0, 2],
        BL['c_o'], BA['ca_c_o'], phi_psi_deg[0, 1] + 180.0,
    )                                                          # O0

    # ── Build residues 1 … L-1 ──────────────────────────────────────────────
    for i in range(1, L):
        phi_i    = phi_psi_deg[i, 0]
        psi_i    = phi_psi_deg[i, 1]
        psi_prev = phi_psi_deg[i - 1, 1]

        n_prev  = coords[i - 1, 0]
        ca_prev = coords[i - 1, 1]
        c_prev  = coords[i - 1, 2]

        # N_i  — dihedral N_{i-1}–CA_{i-1}–C_{i-1}–N_i   = ψ_{i-1}
        n_i  = nerf(n_prev, ca_prev, c_prev,
                    BL['c_n'],  BA['ca_c_n'], psi_prev)

        # CA_i — dihedral CA_{i-1}–C_{i-1}–N_i–CA_i       = ω  (≈180°)
        ca_i = nerf(ca_prev, c_prev, n_i,
                    BL['n_ca'], BA['c_n_ca'], omega_deg)

        # C_i  — dihedral C_{i-1}–N_i–CA_i–C_i            = φ_i
        c_i  = nerf(c_prev, n_i, ca_i,
                    BL['ca_c'], BA['n_ca_c'], phi_i)

        # O_i  — dihedral N_i–CA_i–C_i–O_i                ≈ ψ_i + 180°
        o_i  = nerf(n_i, ca_i, c_i,
                    BL['c_o'],  BA['ca_c_o'], psi_i + 180.0)

        coords[i, 0] = n_i
        coords[i, 1] = ca_i
        coords[i, 2] = c_i
        coords[i, 3] = o_i

    return coords


print("NeRF geometry helpers ready.")

In [ ]:
ONE_TO_THREE = {
    'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU', 'F': 'PHE',
    'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'K': 'LYS', 'L': 'LEU',
    'M': 'MET', 'N': 'ASN', 'P': 'PRO', 'Q': 'GLN', 'R': 'ARG',
    'S': 'SER', 'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
}

BACKBONE_ATOMS = ['N', 'CA', 'C', 'O']


def write_pdb(filepath, sequence, coords, chain_id='A', remark='ESM-2 Flow Matching + NeRF'):
    """
    Write backbone-only coordinates to a PDB file.

    Parameters
    ----------
    filepath  : str, output path (e.g. 'output.pdb')
    sequence  : str of length L, one-letter amino acid codes
    coords    : np.ndarray (L, 4, 3), backbone atoms [N, CA, C, O] in Å
    chain_id  : str, single character chain identifier
    remark    : str, written to REMARK line
    """
    assert len(sequence) == len(coords), \
        f"sequence length {len(sequence)} != coords length {len(coords)}"

    lines = [f"REMARK  {remark}\n"]
    serial = 1

    for res_idx, (aa, res_coords) in enumerate(zip(sequence, coords)):
        res_num  = res_idx + 1
        res_name = ONE_TO_THREE.get(aa, 'UNK')

        for atom_name, (x, y, z) in zip(BACKBONE_ATOMS, res_coords):
            # PDB ATOM record — columns are fixed-width (see PDB format spec)
            name_field = f' {atom_name:<3s}' if len(atom_name) < 4 else atom_name
            line = (
                f"ATOM  "
                f"{serial:5d} "
                f"{name_field:<4s}"
                f"{res_name:>3s} "
                f"{chain_id}"
                f"{res_num:4d}    "
                f"{x:8.3f}{y:8.3f}{z:8.3f}"
                f"  1.00  0.00          "
                f"{atom_name[0]:>2s}\n"
            )
            lines.append(line)
            serial += 1

    lines.append("END\n")

    with open(filepath, 'w') as fh:
        fh.writelines(lines)

    print(f"Wrote {serial - 1} atoms ({len(sequence)} residues) → {filepath}")


print("PDB writer ready.")

In [ ]:
@torch.no_grad()
def generate_structure(
    sequence,
    flow_model,
    esm_model,
    esm_tokenizer,
    n_steps=50,
    omega_deg=180.0,
    output_pdb=None,
    device='cpu',
):
    """
    Full pipeline: sequence → ESM-2 embeddings → φ/ψ angles → NeRF → (PDB).

    Parameters
    ----------
    sequence      : str, amino acid sequence (standard 20 AAs)
    flow_model    : trained ConditionedFlow
    esm_model     : ESM-2 model
    esm_tokenizer : ESM-2 tokenizer
    n_steps       : int, ODE integration steps (more = smoother, default 50)
    omega_deg     : float, peptide bond angle (default 180° = trans)
    output_pdb    : str or None. If provided, writes the structure to this path.
    device        : 'cpu' or 'cuda'

    Returns
    -------
    phi_psi_deg : np.ndarray (L, 2) — generated torsion angles in degrees
    coords      : np.ndarray (L, 4, 3) — backbone [N, CA, C, O] in Å
    """
    L = len(sequence)
    flow_model.eval()
    esm_model.eval()

    # 1. ESM-2 per-residue embeddings ─────────────────────────────────────────
    inputs = esm_tokenizer(sequence, return_tensors='pt').to(device)
    hidden = esm_model(**inputs).last_hidden_state[0]   # (L+2, 320)
    esm2_emb = hidden[1:L + 1].to(device)              # (L, 320) — skip CLS/EOS

    # 2. Flow matching: Gaussian noise → φ/ψ for every residue ───────────────
    x  = torch.randn(L, 2, device=device)
    ts = torch.linspace(0.0, 1.0, n_steps + 1, device=device)

    for i in range(n_steps):
        t_s = ts[i].view(1, 1).expand(L, 1)
        dt  = (ts[i + 1] - ts[i]).float()
        v_s = flow_model(t_s, x, esm2_emb)
        v_m = flow_model(t_s + dt / 2, x + v_s * dt / 2, esm2_emb)
        x   = x + dt * v_m

    phi_psi_deg = x.cpu().numpy() * 180.0   # (L, 2), degrees

    # 3. NeRF: torsion angles → 3D backbone coordinates ──────────────────────
    coords = build_backbone(sequence, phi_psi_deg, omega_deg=omega_deg)

    # 4. Optional PDB export ──────────────────────────────────────────────────
    if output_pdb is not None:
        write_pdb(output_pdb, sequence, coords)

    return phi_psi_deg, coords


print("generate_structure() pipeline ready.")
print("Usage:")
print("  angles, coords = generate_structure(")
print("      sequence, flow, esm_model, tokenizer,")
print("      n_steps=50, output_pdb='my_protein.pdb')")

In [ ]:
# ── Example: generate structure for a held-out test protein ─────────────────
example_pid = sorted(test_proteins)[0]

# Retrieve sequence from cache (all residues, not just interior)
pdb_path = f'PDBs/{example_pid}.pdb'
example_seq, _ = get_residues_with_positions(pdb_path)   # full sequence
print(f"Protein: {example_pid}   Sequence length: {len(example_seq)}")
print(f"Sequence: {example_seq[:60]}{'...' if len(example_seq) > 60 else ''}")

# Run the full pipeline
angles, coords = generate_structure(
    example_seq,
    flow,
    esm_model,
    tokenizer,
    n_steps=50,
    output_pdb=f'{example_pid}_generated.pdb',
)

print(f"\nGenerated angles shape : {angles.shape}  (L × [φ, ψ])")
print(f"Backbone coords shape  : {coords.shape}  (L × [N, CA, C, O] × xyz)")

# ── Quick sanity check: verify bond lengths are correct ─────────────────────
ca_coords = coords[:, 1, :]                          # (L, 3)  Cα only
ca_dists  = np.linalg.norm(np.diff(ca_coords, axis=0), axis=1)  # Cα–Cα distances
print(f"\nCα–Cα distances  mean: {ca_dists.mean():.3f} Å  "
      f"std: {ca_dists.std():.3f} Å  "
      f"(expect ~3.8 Å for folded proteins)")

n_ca_dists = np.linalg.norm(coords[:, 1] - coords[:, 0], axis=1)
ca_c_dists = np.linalg.norm(coords[:, 2] - coords[:, 1], axis=1)
print(f"N–CA bond lengths mean: {n_ca_dists.mean():.3f} Å  (expect 1.458 Å)")
print(f"CA–C  bond lengths mean: {ca_c_dists.mean():.3f} Å  (expect 1.525 Å)")

# ── Ramachandran plot of generated angles ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Generated Structure — {example_pid}', fontsize=13)

axes[0].scatter(angles[:, 0], angles[:, 1], s=20, alpha=0.7, color='tomato')
axes[0].set_title(f'Generated φ/ψ  (L={len(example_seq)})')
axes[0].set_xlabel('φ (°)'); axes[0].set_ylabel('ψ (°)')
axes[0].axhline(0, color='k', lw=0.4, ls='--')
axes[0].axvline(0, color='k', lw=0.4, ls='--')
axes[0].set_xlim(-180, 180); axes[0].set_ylim(-180, 180)

# Cα trace coloured by residue index
ca = coords[:, 1]
sc = axes[1].scatter(ca[:, 0], ca[:, 1], c=np.arange(len(ca)),
                     cmap='plasma', s=15, zorder=3)
axes[1].plot(ca[:, 0], ca[:, 1], lw=0.6, color='grey', alpha=0.5)
axes[1].set_title('Cα trace — XY projection (coloured N→C)')
axes[1].set_xlabel('x (Å)'); axes[1].set_ylabel('y (Å)')
axes[1].set_aspect('equal')
plt.colorbar(sc, ax=axes[1], label='Residue index')

plt.tight_layout()
plt.show()

print(f"\nPDB file written: {example_pid}_generated.pdb")
print("Open in PyMOL / ChimeraX / UCSF Chimera to visualise the 3D structure.")